In [9]:
import sys
import os

from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT))
os.chdir(PROJECT_ROOT)



In [12]:
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch
from pathlib import Path

from src.models.multi_gru import MultiGRUForecast
from src.models.hgru import HierarchicalGRUForecast
from src.training.setup import setup_experiment, get_dataloaders
from src.training.evaluate import evaluate_model
from src.utils.scale import inverse_target

ctx = setup_experiment("config.yaml", "plotting", True)
cfg = ctx["cfg"]
device = ctx["device"]
scaler = ctx["scaler"]
numeric_cols = ctx["numeric_cols"]
target_cols = cfg["data"]["target_cols"]


print(f"Target columns: {target_cols}")

[device] Using MPS
Target columns: ['nitrogen_dioxide', 'ozone', 'pm10', 'pm2_5']


In [14]:
# Get dataloaders (set multi_target=True if using MultiGRU/HGRU)
train_loader, val_loader, test_loader = get_dataloaders(cfg, multi_target=True, multi_city=False)

# Inspect one batch
x, y = next(iter(test_loader))
print(f"Input shape: {x.shape} (Batch, Length, Features)")
print(f"Target shape: {y.shape} (Batch, Horizon, Targets)")

TypeError: get_dataloaders() got an unexpected keyword argument 'multi_city'

In [ ]:
# --- Configuration ---
model_type = "MultiGRU"  # or "HGRU"
results_dir = Path("results")

# --- Initialize Model ---
data_cfg = cfg["data"]
input_size = len(data_cfg["feature_cols"])
horizon = data_cfg["horizon"]

if model_type == "MultiGRU":
    # These params should ideally match what you trained with
    # You might want to load these from a saved MLflow run or a separate config
    model = MultiGRUForecast(
        input_size=input_size,
        target_cols=target_cols,
        horizon=horizon,
        shared_hidden_size=112, 
        branch_hidden_size=64,
        num_layers=1,
        dropout=0.44
    ).to(device)
    best_model_path = results_dir / "multi_gru_best.pt"

elif model_type == "HGRU":
    hgru_cfg = cfg["models"].get("hgru", {})
    model = HierarchicalGRUForecast(
        input_size=input_size,
        target_cols=target_cols,
        horizon=horizon,
        downsample_factor=hgru_cfg.get("downsample_factor", 24),
        short_hidden_size=hgru_cfg.get("short_hidden_size", 64),
        long_hidden_size=hgru_cfg.get("long_hidden_size", 32),
        num_layers_short=hgru_cfg.get("num_layers_short", 1),
        num_layers_long=hgru_cfg.get("num_layers_long", 1),
        dropout=hgru_cfg.get("dropout", 0.3)
    ).to(device)
    best_model_path = results_dir / "hier_hgru_best.pt"

# --- Load Weights ---
print(f"Loading weights from {best_model_path}...")
model.load_state_dict(torch.load(best_model_path, map_location=device))
model.eval()

In [ ]:
# Run inference
y_true_all, y_pred_all = [], []

with torch.no_grad():
    for x, y in test_loader:
        x, y = x.to(device), y.to(device)
        y_hat = model(x)
        y_true_all.append(y.cpu())
        y_pred_all.append(y_hat.cpu())

y_true = torch.cat(y_true_all, dim=0)
y_pred = torch.cat(y_pred_all, dim=0)

# Select specific target to plot
plot_target = "nitrogen_dioxide"
t_idx = target_cols.index(plot_target)

# Slice and Denormalize
y_true_target = y_true[..., t_idx] # [N, Horizon]
y_pred_target = y_pred[..., t_idx]

y_true_den = inverse_target(scaler, y_true_target, numeric_cols, plot_target)
y_pred_den = inverse_target(scaler, y_pred_target, numeric_cols, plot_target)

print(f"Denormalized shapes: {y_true_den.shape}")

In [ ]:
# --- Plot Parameters ---
sample_idx = 0  # Change this to see different samples
horizon_steps = torch.arange(horizon)

fig, ax = plt.subplots(figsize=(10, 5))

# Plot ground truth vs prediction for a single sample
ax.plot(horizon_steps, y_true_den[sample_idx].numpy(), label="Actual", marker='o')
ax.plot(horizon_steps, y_pred_den[sample_idx].numpy(), label="Predicted", marker='x', linestyle='--')

ax.set_title(f"Forecast for {plot_target} (Sample {sample_idx})")
ax.set_xlabel("Horizon Steps")
ax.set_ylabel(plot_target)
ax.legend()
ax.grid(True, alpha=0.3)

plt.show()